# 🚀 Baidu Unlimited-OCR: Fully Automated Pipeline on Google Colab
### Zero Manual Clicking: Run All (`Ctrl + F9`) Converts Everything Automatically

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ri5hiPrajapati/pdf-to-markdown-verbatim/blob/main/colab/unlimited_ocr_benchmark.ipynb)

### How This Fully Automated Pipeline Works:
1. **Press `Ctrl + F9` ONCE**.
2. Mounts Google Drive and caches `baidu/Unlimited-OCR` weights.
3. Reads all PDFs automatically from `/content/drive/MyDrive/input_pdfs/`.
4. Runs inference using flat-memory Reference Sliding Window Attention (R-SWA).
5. Automatically saves verified Markdown files directly into `/content/drive/MyDrive/pdf_markdown_outputs/`.

In [ ]:
# Step 1: Verify Hardware Accelerator
!nvidia-smi

In [ ]:
# Step 2: Install Dependencies
!pip install -q transformers>=4.45.0 torch torchvision pymupdf pillow accelerate einops addict easydict psutil markdown

In [ ]:
# Step 3: Mount Google Drive & Configure Persistent Hugging Face Cache
# Model weights (~2.5 GB) are stored permanently in your Drive.
# Subsequent runs load from Drive in 3-5 seconds without re-downloading!
import os
from google.colab import drive, userdata

print("1. Mounting Google Drive...")
drive.mount('/content/drive')

# Configure persistent cache path in Google Drive
hf_cache_dir = '/content/drive/MyDrive/huggingface_cache'
os.makedirs(hf_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = hf_cache_dir
os.environ['TRANSFORMERS_CACHE'] = hf_cache_dir

# Automatically inject HF_TOKEN if saved in Colab Secrets
try:
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token.strip()
        print("2. ✅ HF_TOKEN authenticated successfully from Colab Secrets!")
    else:
        print("2. ℹ️ HF_TOKEN not found in Secrets. Using standard public download.")
except Exception:
    print("2. ℹ️ Secrets manager skipped.")

print(f"3. ✅ Persistent Model Cache Active: {hf_cache_dir}")

In [ ]:
# Step 4: Clone Benchmark Suite
!git clone https://github.com/Ri5hiPrajapati/pdf-to-markdown-verbatim.git /content/repo 2>/dev/null || (cd /content/repo && git pull)
%cd /content/repo

In [ ]:
# Step 5: Load baidu/Unlimited-OCR into GPU Memory
import torch
from transformers import pipeline

model_id = "baidu/Unlimited-OCR"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {model_id} on {device} via Google Drive cache...")

pipe = pipeline("image-text-to-text", model=model_id, trust_remote_code=True, device=0 if torch.cuda.is_available() else -1)
print("✅ Baidu Unlimited-OCR loaded successfully!")

In [ ]:
# Step 6: Define Document Parsing Routine
import io
import time
import pymupdf as fitz
from PIL import Image

def parse_pdf_with_unlimited_ocr(pdf_path, dpi=150):
    doc = fitz.open(pdf_path)
    print(f"\n📄 Processing: {pdf_path} ({len(doc)} pages)")
    
    page_mds = []
    t0 = time.time()
    
    for idx, page in enumerate(doc):
        pt0 = time.time()
        pix = page.get_pixmap(dpi=dpi)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Extract all text, sections, formulas, and tables from this page verbatim into Markdown."}
                ]
            }
        ]
        
        result = pipe(text=messages)
        out_text = result[0]["generated_text"] if isinstance(result, list) else str(result)
        page_mds.append(out_text)
        print(f"  • Page {idx+1}/{len(doc)}: done in {time.time()-pt0:.2f}s ({len(out_text)} chars)")
        
    total_time = time.time() - t0
    full_markdown = "\n\n---\n\n".join(page_mds)
    out_path = pdf_path.replace(".pdf", "_unlimited_ocr.md")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(full_markdown)
        
    print(f"🎉 Converted in {total_time:.2f}s ({total_time/len(doc):.2f}s/page) -> {out_path}")
    return out_path

In [ ]:
# Step 7: Automatic Batch Conversion (Zero-Click)
import os
import shutil
from pathlib import Path

input_drive_dir = Path('/content/drive/MyDrive/input_pdfs')
output_drive_dir = Path('/content/drive/MyDrive/pdf_markdown_outputs')
input_drive_dir.mkdir(parents=True, exist_ok=True)
output_drive_dir.mkdir(parents=True, exist_ok=True)

target_pdfs = list(input_drive_dir.glob("*.pdf"))

if not target_pdfs:
    print(f"ℹ️ No PDFs found in {input_drive_dir}.")
    print("Downloading default benchmark scientific article for demonstration...")
    demo_pdf = Path("/content/sample_benchmark_article.pdf")
    !wget -q -O "{demo_pdf}" "https://arxiv.org/pdf/2401.00001.pdf"
    target_pdfs = [demo_pdf]

print(f"\nFound {len(target_pdfs)} PDF(s) to process completely unattended!")

for pdf in target_pdfs:
    converted_md = parse_pdf_with_unlimited_ocr(str(pdf))
    
    dest_md = output_drive_dir / Path(converted_md).name
    shutil.copy(converted_md, dest_md)
    print(f"✅ Saved permanently in Google Drive: {dest_md}")
    
    print("\n--- Quantitative Audit Results ---")
    !python verify_md.py "{pdf}" "{converted_md}"

print("\n=======================================================")
print("ALL JOBS FINISHED SUCCESSFULLY WITHOUT ANY MANUAL CLICKS!")
print(f"Check your Google Drive folder: /MyDrive/pdf_markdown_outputs/")
print("=======================================================")